## 1、数据处理

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from collections import Counter

### 1.1 导入数据

In [22]:
# 加载数据
data_path = './data/news.csv'  # 更新为正确的路径
data = pd.read_csv(data_path)

In [23]:
data

,label,text
0,教育,澳移民子女成长记：带着中国心融入主流社会新华网悉尼5月31日电 无论哪个国家的父辈与子女间都...
1,体育,快船vs火箭首发：休城旨在练兵 小德帕特森进先发新浪体育讯北京时间4月10日消息，在常规赛还...
2,科技,3英寸屏高清闪存DV 三洋TH1特价1499 作者：中关村在线 飘雪 ...
3,时尚,贝嫂乏味归乏味 还有人买账Victoria Beckham 乏味归乏味 还有人买账大姐大的阵...
4,房产,三亚岭南赶房集 金九银十再兴购房游纯粹的旅行闲适有余却“收获”不足，设计一条可以兼容曼妙风景...
...,...,...
995,娱乐,组图：本-斯蒂勒与布莱克助阵《寻找伴郎》首映新浪娱乐讯 北京时间3月18日(美国当地时间3月...
996,房产,"房源阶段性不足价格高涨 楼市将上演新一轮疯狂近日，中海地产(企业专区,旗下楼盘)以70.06..."
997,科技,宽屏广角高清DC！佳能110IS仅售1980【山东IT在线报道】佳能IXUS 110 IS装...
998,时政,公安部建成打拐DNA数据库通缉50名人贩●不到1个月，侦破拐卖儿童、妇女案件逾300起，解救...


In [24]:
data['label'].unique()

array(['教育', '体育', '科技', '时尚', '房产', '家居', '财经', '时政', '娱乐', '游戏'],
      dtype=object)

### 1.2 创建词汇表

In [25]:
# 1. 数据准备
class Vocabulary:
    def __init__(self, freq_threshold):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {v: k for k, v in self.itos.items()}
        self.freq_threshold = freq_threshold

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4

        for sentence in sentence_list:
            for word in sentence:
                frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        return [self.stoi[token] if token in self.stoi else self.stoi["<UNK>"] for token in text]

class NewsDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts.iloc[index]
        label = self.labels.iloc[index]
        numericalized_text = [self.vocab.stoi["<SOS>"]] + self.vocab.numericalize(text)[:self.max_length-2] + [self.vocab.stoi["<EOS>"]]
        padded_text = numericalized_text + [self.vocab.stoi["<PAD>"]] * (self.max_length - len(numericalized_text))
        return torch.tensor(padded_text, dtype=torch.long), torch.tensor(label, dtype=torch.long)

In [26]:
vocab = Vocabulary(freq_threshold=5)
vocab.build_vocabulary(data['text'].apply(list))

In [27]:
vocab.itos

{0: '<PAD>',
 1: '<SOS>',
 2: '<EOS>',
 3: '<UNK>',
 4: '，',
 5: '中',
 6: '国',
 7: '的',
 8: '澳',
 9: '大',
 10: '对',
 11: '不',
 12: '。',
 13: '民',
 14: '子',
 15: '珍',
 16: '妮',
 17: '父',
 18: '学',
 19: '女',
 20: '语',
 21: '和',
 22: '文',
 23: '华',
 24: '社',
 25: '会',
 26: '、',
 27: '化',
 28: '是',
 29: '在',
 30: '融',
 31: '入',
 32: '主',
 33: '流',
 34: '一',
 35: '他',
 36: '同',
 37: '人',
 38: '新',
 39: '利',
 40: '亚',
 41: '们',
 42: '母',
 43: '家',
 44: '教',
 45: '育',
 46: '要',
 47: '生',
 48: '说',
 49: '代',
 50: '以',
 51: '系',
 52: '关',
 53: '时',
 54: '快',
 55: '有',
 56: '发',
 57: '：',
 58: '体',
 59: ' ',
 60: '1',
 61: '价',
 62: '9',
 63: '0',
 64: '所',
 65: '下',
 66: '这',
 67: '了',
 68: '常',
 69: '机',
 70: '为',
 71: '3',
 72: '5',
 73: '重',
 74: '三',
 75: '种',
 76: '者',
 77: '2',
 78: 'D',
 79: '还',
 80: '用',
 81: '英',
 82: '寸',
 83: '等',
 84: '4',
 85: '最',
 86: 'm',
 87: '感',
 88: '光',
 89: '像',
 90: '能',
 91: '值',
 92: '洋',
 93: 'T',
 94: 'H',
 95: '可',
 96: '7',
 97: '.',
 98: '摄',
 99:

In [28]:
vocab.stoi

{'<PAD>': 0,
 '<SOS>': 1,
 '<EOS>': 2,
 '<UNK>': 3,
 '，': 4,
 '中': 5,
 '国': 6,
 '的': 7,
 '澳': 8,
 '大': 9,
 '对': 10,
 '不': 11,
 '。': 12,
 '民': 13,
 '子': 14,
 '珍': 15,
 '妮': 16,
 '父': 17,
 '学': 18,
 '女': 19,
 '语': 20,
 '和': 21,
 '文': 22,
 '华': 23,
 '社': 24,
 '会': 25,
 '、': 26,
 '化': 27,
 '是': 28,
 '在': 29,
 '融': 30,
 '入': 31,
 '主': 32,
 '流': 33,
 '一': 34,
 '他': 35,
 '同': 36,
 '人': 37,
 '新': 38,
 '利': 39,
 '亚': 40,
 '们': 41,
 '母': 42,
 '家': 43,
 '教': 44,
 '育': 45,
 '要': 46,
 '生': 47,
 '说': 48,
 '代': 49,
 '以': 50,
 '系': 51,
 '关': 52,
 '时': 53,
 '快': 54,
 '有': 55,
 '发': 56,
 '：': 57,
 '体': 58,
 ' ': 59,
 '1': 60,
 '价': 61,
 '9': 62,
 '0': 63,
 '所': 64,
 '下': 65,
 '这': 66,
 '了': 67,
 '常': 68,
 '机': 69,
 '为': 70,
 '3': 71,
 '5': 72,
 '重': 73,
 '三': 74,
 '种': 75,
 '者': 76,
 '2': 77,
 'D': 78,
 '还': 79,
 '用': 80,
 '英': 81,
 '寸': 82,
 '等': 83,
 '4': 84,
 '最': 85,
 'm': 86,
 '感': 87,
 '光': 88,
 '像': 89,
 '能': 90,
 '值': 91,
 '洋': 92,
 'T': 93,
 'H': 94,
 '可': 95,
 '7': 96,
 '.': 97,
 '摄': 98,
 '出'

### 1.3 数据划分

In [29]:
label_to_int = {label: index for index, label in enumerate(data['label'].unique())}
data['label'] = data['label'].map(label_to_int)
train_texts, test_texts, train_labels, test_labels = train_test_split(data['text'], data['label'], test_size=0.2)

### 1.4 创建DataLoader

In [30]:
max_length = 256
train_dataset = NewsDataset(train_texts, train_labels, vocab, max_length)
test_dataset = NewsDataset(test_texts, test_labels, vocab, max_length)
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

## 2、创建模型

### 2.1 定义模型

In [54]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        print("self.embedding: ",self.embedding)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=n_layers, bidirectional=bidirectional, dropout=dropout, batch_first=True)
        print("self.rnn: ",self.rnn)
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        print("self.fc: ",self.fc)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        #print("embedded: ",self.embedding(text).shape)
        embedded = self.dropout(self.embedding(text))
        print("embedded: ",embedded.shape)
        output, hidden = self.rnn(embedded)
        print("output: ",output.shape)
        print("hidden: ",hidden.shape)
        if self.rnn.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden = self.dropout(hidden[-1,:,:])
            
        print("hidden.shape: ",hidden.shape)
        return self.fc(hidden)

### 2.2 模型超参数

In [55]:
# 模型参数
vocab_size = len(vocab.stoi)
embedding_dim = 100
hidden_dim = 256
output_dim = len(label_to_int)
n_layers = 2
bidirectional = False
dropout = 0.5
model = RNN(vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout)

self.embedding:  Embedding(2970, 100)
self.rnn:  RNN(100, 256, num_layers=2, batch_first=True, dropout=0.5)
self.fc:  Linear(in_features=256, out_features=10, bias=True)


In [56]:
next(iter(train_loader))

[tensor([[   1,  224,  178,  ...,  223,  100,    2],
         [   1,  510,  729,  ...,  788,  190,    2],
         [   1,  183,  513,  ...,  735,    4,    2],
         ...,
         [   1,   85,   38,  ...,  764,  953,    2],
         [   1,  630,  833,  ...,    0,    0,    0],
         [   1,    9, 1005,  ..., 1109,  764,    2]]),
 tensor([6, 0, 4, 2, 2, 9, 8, 3, 9, 2, 6, 1, 3, 2, 8, 5, 3, 2, 3, 0, 3, 5, 4, 9,
         7, 1, 4, 6, 2, 5, 4, 0, 4, 5, 8, 4, 7, 0, 5, 0, 9, 3, 8, 8, 7, 3, 7, 3,
         2, 7, 7, 3, 0, 6, 7, 9, 2, 3, 8, 6, 6, 0, 3, 9])]

In [57]:
next(iter(train_loader))[0].shape, next(iter(train_loader))[1].shape

(torch.Size([64, 256]), torch.Size([64]))

In [58]:
model(next(iter(train_loader))[0])

embedded:  torch.Size([64, 256, 100])
embedded:  torch.Size([64, 256, 100])
output:  torch.Size([64, 256, 256])
hidden:  torch.Size([2, 64, 256])
hidden.shape:  torch.Size([64, 256])


tensor([[-1.5244e-01, -1.3066e-01,  6.5238e-04,  3.1577e-01,  1.8758e-01,
          1.0281e-01,  1.7963e-02, -3.3357e-01,  2.1537e-01, -1.9987e-02],
        [ 6.9615e-02, -1.8990e-01, -1.2086e-01, -8.3715e-02, -2.8309e-01,
          5.1199e-01, -3.4535e-01,  3.4439e-01, -4.1855e-01,  1.7480e-01],
        [-2.5374e-01, -1.8851e-01, -3.1983e-01, -6.1445e-02, -2.3172e-01,
         -1.8382e-01, -7.3593e-02, -4.5055e-01, -4.4665e-02, -1.0248e-01],
        [ 2.2184e-01, -3.5306e-01,  2.0986e-01,  1.9579e-01,  1.8030e-01,
          7.9642e-02, -5.2310e-01,  3.3014e-01,  4.7146e-01, -2.1188e-01],
        [ 2.7181e-01,  1.2479e-01, -5.6331e-01, -4.5930e-01, -2.9860e-01,
          1.7744e-01, -3.7300e-01, -2.0743e-01, -1.9322e-01, -3.0081e-01],
        [-3.1459e-01, -4.7743e-01,  2.3072e-01,  4.0578e-02,  2.6222e-01,
         -3.8168e-01,  3.5283e-01, -8.9289e-02, -4.0107e-01, -5.3191e-02],
        [ 3.4378e-01,  3.7474e-02,  4.4437e-01,  1.1355e-01,  5.7602e-01,
         -1.9833e-01,  3.4341e-0

### 2.3 配置模型

In [36]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=n_layers, bidirectional=bidirectional, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        output, hidden = self.rnn(embedded)
        if self.rnn.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden = self.dropout(hidden[-1,:,:])
        return self.fc(hidden)

In [50]:
# 模型参数
vocab_size = len(vocab.stoi)
embedding_dim = 100
hidden_dim = 256
output_dim = len(label_to_int)
n_layers = 2
bidirectional = True
dropout = 0.6
model = RNN(vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout)

In [51]:
# 3. 训练模型
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = criterion.to(device)

### 2.4 定义训练函数

In [52]:
def train(model, iterator, optimizer, criterion):
    model.train()
    epoch_loss = 0
    epoch_acc = 0

    for batch in iterator:
        optimizer.zero_grad()
        predictions = model(batch[0].to(device))
        loss = criterion(predictions, batch[1].to(device))
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        # 计算准确率
        _, preds = torch.max(predictions, dim=1)
        epoch_acc += torch.sum(preds == batch[1].to(device)).item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator.dataset)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    epoch_acc = 0

    with torch.no_grad():
        for batch in iterator:
            predictions = model(batch[0].to(device))
            loss = criterion(predictions, batch[1].to(device))
            epoch_loss += loss.item()
            # 计算准确率
            _, preds = torch.max(predictions, dim=1)
            epoch_acc += torch.sum(preds == batch[1].to(device)).item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator.dataset)



### 2.5 开始训练

In [53]:
num_epochs = 20
for epoch in range(num_epochs):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, test_loader, criterion)
    
    print(f'Epoch: {epoch+1}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val. Loss: {valid_loss:.4f}, Val. Acc: {valid_acc:.4f}')

Epoch: 1, Train Loss: 2.3643, Train Acc: 0.1313, Val. Loss: 2.0167, Val. Acc: 0.3650
Epoch: 2, Train Loss: 2.1719, Train Acc: 0.2200, Val. Loss: 1.8456, Val. Acc: 0.4500
Epoch: 3, Train Loss: 2.0552, Train Acc: 0.2687, Val. Loss: 1.7641, Val. Acc: 0.4850
Epoch: 4, Train Loss: 1.9454, Train Acc: 0.2988, Val. Loss: 1.6830, Val. Acc: 0.4800
Epoch: 5, Train Loss: 1.8321, Train Acc: 0.3750, Val. Loss: 1.5754, Val. Acc: 0.5050
Epoch: 6, Train Loss: 1.7674, Train Acc: 0.3725, Val. Loss: 1.5361, Val. Acc: 0.5200
Epoch: 7, Train Loss: 1.6132, Train Acc: 0.4462, Val. Loss: 1.4563, Val. Acc: 0.5450
Epoch: 8, Train Loss: 1.5182, Train Acc: 0.4900, Val. Loss: 1.8697, Val. Acc: 0.4250
Epoch: 9, Train Loss: 1.4809, Train Acc: 0.5000, Val. Loss: 1.2934, Val. Acc: 0.5750
Epoch: 10, Train Loss: 1.3723, Train Acc: 0.5413, Val. Loss: 1.3452, Val. Acc: 0.5250
Epoch: 11, Train Loss: 1.2773, Train Acc: 0.5425, Val. Loss: 1.3153, Val. Acc: 0.5700
Epoch: 12, Train Loss: 1.2564, Train Acc: 0.5763, Val. Loss: 1.